# 02. 前処理: 欠損値の扱いと、カテゴリ変数の渡し方

**このノートブックの目的:** モデルに渡す前に、欠損値とカテゴリ変数の型をどう扱うかを決める。

**結論:** 欠損はArrival Delay in Minutesの0.04%だけで、補完しない。カテゴリ変数4つはstr型からcategory型に変換してLightGBMに渡す。

## 準備

- ライブラリと、共通処理(`src/`の自分で書いたファイル)を読み込む。`src/features.py` はデータの読み込みと特徴量の作成、`src/modeling.py` はモデルの学習と評価。
- `DATA`: train.csv・test.csvを置き、予測も保存するフォルダ(`data/`)の場所。
- train・testを読み込み、目的変数 `satisfaction` を0/1の配列 `y` にする。

In [1]:
import sys  # Pythonの実行環境の設定(下の sys.path)に使う

sys.path.insert(0, "../src")  # src/ を読み込み先に加え、features.py・modeling.py を import できるようにする
import matplotlib.pyplot as plt  # グラフを描く
import numpy as np  # 数値の配列の計算
import pandas as pd  # 表の読み込みと集計
from sklearn.metrics import roc_auc_score  # 評価指標のAUCを計算する

import features  # src/features.py: データの読み込みと特徴量の作成
import modeling  # src/modeling.py: モデルの学習と評価

plt.rcParams["font.family"] = "Meiryo"  # グラフの日本語ラベル用
DATA = features.DATA_DIR  # train.csv・test.csvを置き、予測も保存するフォルダ(data/)の場所
train, test = features.load_train_and_test()  # data/ の train.csv・test.csv を読み込む
y = train[features.TARGET].astype(int).values  # 目的変数 satisfaction(満足=True)を0/1の配列にする
print(f"train {len(train):,}行・test {len(test):,}行、満足率 {y.mean():.3f}")  # 行数と満足の割合を確かめる

train 699,635行・test 299,844行、満足率 0.444


## 1. 欠損値の扱い

#### ① 目的・仮説
- **確認したいこと:** 欠損がある特徴量はどれか。補完する必要があるか。
- **なぜ必要か:** 欠損が多ければ、補完の仕方でスコアが変わりうる。補完するかどうかは、学習の前に決めておく必要がある。
- **想定する結果:** 人工データなので、欠損は少なく、補完しなくてもよい。

#### ② 検証方法
- **データ:** trainとtest。
- **方法:** 特徴量ごとに、欠損(NaN)の件数と割合を数える。

#### ③ コード・実行結果

In [2]:
missing = pd.DataFrame({"train": train.isna().sum(), "test": test.isna().sum()}).fillna(0).astype(int)
missing["trainの割合"] = (missing["train"] / len(train)).map("{:.2%}".format)
missing[missing[["train", "test"]].sum(axis=1) > 0]

,train,test,trainの割合
Arrival Delay in Minutes,292,130,0.04%


#### ④ 考察・次のアクション
- **読み取れること:** 欠損はArrival Delay in Minutesだけで、train 292行・test 130行(どちらも0.04%)。
- **想定との比較:** 想定どおり、欠損は少なかった。
- **次のアクション:**
  - 欠損は0.04%しかなく、補完してもAUCが上がる見込みは薄い。LightGBMは欠損をそのまま扱えるので、補完せずに進めた(00のベースラインも補完していない)。
  - 念のため、特徴量を作ったあとの土台で、欠損を同じ人のDeparture Delay in Minutesで埋めて比べた(欠損していない行では、Departure Delay in MinutesとArrival Delay in Minutesの相関0.84、86.9%が完全に一致)。結果は3seed平均−0.000009でほぼ効果がなく、採用しなかった。
  - サービス評価の0点は、値は入っていても「評価なし」の可能性がある。これは01_EDAの7節で確かめた。

## 2. カテゴリ変数の渡し方

#### ① 目的・仮説
- **確認したいこと:** Gender, Customer Type, Type of Travel, ClassがLightGBMに渡せる型であるか。trainにない値がtestにあるか。
- **なぜ必要か:** LightGBMはstr型を受け取れないので、category型に変える必要がある。testにだけある値があると、予測のときに扱いが変わる。
- **想定する結果:** 読み込んだ時点はstr型で、そのままでは渡せない。trainとtestで値はそろっている。

#### ② 検証方法
- **データ:** trainとtest。
- **方法:** 読み込んだ時点の型を確かめる。カテゴリ変数ごとに、値の種類と件数をtrainとtestで並べる。

#### ③ コード・実行結果

In [3]:
print("読み込んだ時点の型:", train[features.CATEGORICAL_COLUMNS].dtypes.astype(str).to_dict())
for c in features.CATEGORICAL_COLUMNS:
    counts = pd.DataFrame({"train": train[c].value_counts(), "test": test[c].value_counts()})
    print(f"{c}: {len(counts)}種類 / testにだけある値: {"、".join(sorted(set(test[c].dropna()) - set(train[c].dropna()))) or "なし"}")
    print(counts.to_string(), "\n")

読み込んだ時点の型: {'Gender': 'str', 'Customer Type': 'str', 'Type of Travel': 'str', 'Class': 'str'}


Gender: 2種類 / testにだけある値: なし
         train    test
Gender                
Male    351683  150261
Female  347952  149583 



Customer Type: 2種類 / testにだけある値: なし
                    train    test
Customer Type                    
Loyal Customer     576990  247708
disloyal Customer  122645   52136 



Type of Travel: 2種類 / testにだけある値: なし
                  train    test
Type of Travel                 
Business travel  497441  213141
Personal Travel  202194   86703 



Class: 3種類 / testにだけある値: なし
           train    test
Class                   
Business  342212  146626
Eco       327404  140220
Eco Plus   30019   12998 



#### ④ 考察・次のアクション
- **読み取れること:** 読み込んだ時点の型は4つともstr型。testにだけある値はない。
- **想定との比較:** 想定どおりだった。
- **次のアクション:** str型からcategory型に変換してLightGBMに渡す。

## Summary

| 決めたこと | 値 |
|---|---|
| 欠損値 | 補完しない(Arrival Delay in Minutesの0.04%だけ。Departure Delay in Minutesで埋めても−0.000009) |
| カテゴリ変数 | str型からcategory型に変換して渡す |